# FINRA短卖标记构成：第一阶段来源资格

本入口只核对冻结日历、样本与HTTP预算，不读取金融标签或执行模型。未来唯一q提议及B/C完整Stage1对照在experiment_design_draft.json，尚未批准。

In [1]:
import os, sys, json, hashlib
from pathlib import Path
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root: raise RuntimeError("workspace root not found")
    root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import qlib
from qlib.data import D
from qlib.workflow import R
card = root / "research/stage1_finra_short_mix_20261018"
plan = json.loads((card / "plan.json").read_text())
calendar_binding = plan["calendar"]
calendar_path = root / calendar_binding["calendar_file"]
assert hashlib.sha256(calendar_path.read_bytes()).hexdigest() == calendar_binding["calendar_file_sha256"]
qlib.init(provider_uri=str(root / calendar_binding["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(start_time="2018-08-01", end_time="2026-10-09", freq="day")
assert len(calendar) == 2059
{"calendar_rows": len(calendar), "maximum_HTTP_requests": plan["requests"]["maximum_total_HTTP_requests"], "model_fits": 0, "accounts": 0}


[53562:MainThread](2026-10-18 12:02:42,358) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[53562:MainThread](2026-10-18 12:02:42,360) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[53562:MainThread](2026-10-18 12:02:42,360) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'calendar_rows': 2059,
 'maximum_HTTP_requests': 2242,
 'model_fits': 0,
 'accounts': 0}

## 有限来源采集

正式批量HTTP采集等待独立design_review与owner源采集执行指令。按source_calendar逐日一次请求；严格保留原始响应、当前接收时刻、版本标签与缺口。研究发布时钟假设不升级为真实历史PIT资格。

In [2]:
json.loads((card / "source_inventory.json").read_text())


{'stage': 'four_raw_readiness_samples_retained_bulk_not_started',
 'calendar_count': 2059,
 'month_index_count': 99,
 'budget_HTTP_max': 2242,
 'probe_receipts_sha256': '3c9adb08ebe149fa26d13a48d8d43d46c04cc1959d257dee69d883266972cafc',
 'probe_samples': [{'trade_date': '20180801',
   'url': 'https://cdn.finra.org/equity/regsho/daily/CNMSshvol20180801.txt',
   'sha256': '7f207fe39c7c5c69cd6a92f15d8a6fd5990742279d26da6a9c27d165834dd79e',
   'bytes': 249497,
   'raw_temporary_path': '/tmp/stage1-finra-short-mix-source-probes-20261018/20180801.txt',
   'received_at': '2026-10-18T03:59:12.343948+00:00',
   'historical_original_vintage_certified': False},
  {'trade_date': '20200316',
   'url': 'https://cdn.finra.org/equity/regsho/daily/CNMSshvol20200316.txt',
   'sha256': '53c28a040247936c68a947ab6fa8a71298665806566882f50f0f75ca207af467',
   'bytes': 291877,
   'raw_temporary_path': '/tmp/stage1-finra-short-mix-source-probes-20261018/20200316.txt',
   'received_at': '2026-10-18T03:59:13.971

In [3]:
from IPython.display import Markdown, display
display(Markdown((card / "conclusion.md").read_text()))


# FINRA场外卖空标记构成：来源资格准备

当前只冻结来源采集。原生Qlib日历包含2018-08-01至2026-10-09共2059个日期；采集硬预算2242个HTTP请求，包含99个月索引、4份官方文档、至多64个明确Updated文件和16次网络失败重试。4日原始TXT已保存临时来源对象与SHA、真实当前接收时刻；这不是历史首次vintage认证。

ShortVolume已含ShortExemptVolume，不能再相加。CNMS仅是场外公开报告交易构成，不是short interest、净库存或情绪。发行人SOXX旧workbook没有同期市价，已放弃从其NAV拼造折溢价。

0拟合、0标签计算、0账户，不晋升，生产未改。下一步是独立评估本source-only计划后的一次有预算历史采集；6-fit Stage1 q增量模型仍是单独未批准草案。来源完成不等于稳定超额目标达到。
